# Tune LightBGM

| Parameter           | Controls         | Risk                |
| ------------------- | ---------------- | ------------------- |
| `num_leaves`        | Model complexity | Overfitting         |
| `max_depth`         | Tree depth limit | Over/underfit       |
| `learning_rate`     | Step size        | Too high = unstable |
| `n_estimators`      | Number of trees  | Overfit if too high |
| `min_child_samples` | Regularization   | Too high = underfit |
| `subsample`        | Row sampling     | Too low = underfit |

In [1]:
import pandas as pd

train = pd.read_csv("../data/processed/train.csv")
test  = pd.read_csv("../data/processed/test.csv")

X_train = train.drop(columns=["Churn"])
y_train = train["Churn"]

X_test  = test.drop(columns=["Churn"])
y_test  = test["Churn"]

# Drop CustomerID column as it is not useful for modeling
X_train = X_train.drop(columns=["CustomerID"])
X_test  = X_test.drop(columns=["CustomerID"])

In [2]:
# Preprocessing pipeline

numeric_features = X_train.select_dtypes(include=["int64","float64"]).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object"]).columns.tolist()

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median", add_indicator=True))
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

from sklearn.compose import ColumnTransformer
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

In [ ]:
# Evalution Function
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix

def evaluate_model(model, X_train, y_train, X_test, y_test):
    
    model.fit(X_train, y_train)
    
    y_train_pred = model.predict_proba(X_train)[:,1]
    y_test_pred  = model.predict_proba(X_test)[:,1]
    
    train_auc = roc_auc_score(y_train, y_train_pred)
    test_auc  = roc_auc_score(y_test, y_test_pred)
    
    print(f"Train ROC-AUC: {train_auc:.4f}")
    print(f"Test ROC-AUC : {test_auc:.4f}")
    
    y_test_class = (y_test_pred >= 0.5).astype(int)
    
    print("\nClassification Report:")
    print(classification_report(y_test, y_test_class))
    
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, y_test_class))
    
    return test_auc

### 1. Try num_leaves with = [15, 31, 50, 70]

In [8]:
# LightBGM model
from lightgbm import LGBMClassifier
from sklearn.pipeline import Pipeline

lightBGM = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LGBMClassifier(
        class_weight="balanced",
        num_leaves = 15,
        random_state=42
    ))
])

In [9]:
evaluate_model(lightBGM, X_train, y_train, X_test, y_test)

[LightGBM] [Info] Number of positive: 6981, number of negative: 33019
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.010840 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1550
[LightGBM] [Info] Number of data points in the train set: 40000, number of used features: 72
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000


C:\Users\vedik\AppData\Roaming\Python\Python311\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Train ROC-AUC: 0.8130
Test ROC-AUC : 0.7860

Classification Report:
              precision    recall  f1-score   support

           0       0.92      0.74      0.82      8255
           1       0.36      0.69      0.47      1745

    accuracy                           0.73     10000
   macro avg       0.64      0.71      0.64     10000
weighted avg       0.82      0.73      0.76     10000


Confusion Matrix:
[[6083 2172]
 [ 547 1198]]


C:\Users\vedik\AppData\Roaming\Python\Python311\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


np.float64(0.7859561713921752)

In [10]:
lightBGM = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LGBMClassifier(
        class_weight="balanced",
        num_leaves = 31,
        random_state=42
    ))
])
evaluate_model(lightBGM, X_train, y_train, X_test, y_test)

[LightGBM] [Info] Number of positive: 6981, number of negative: 33019
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.008532 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1550
[LightGBM] [Info] Number of data points in the train set: 40000, number of used features: 72
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000


C:\Users\vedik\AppData\Roaming\Python\Python311\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Train ROC-AUC: 0.8490
Test ROC-AUC : 0.7802

Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.75      0.82      8255
           1       0.36      0.66      0.46      1745

    accuracy                           0.74     10000
   macro avg       0.64      0.70      0.64     10000
weighted avg       0.82      0.74      0.76     10000


Confusion Matrix:
[[6209 2046]
 [ 600 1145]]


C:\Users\vedik\AppData\Roaming\Python\Python311\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


np.float64(0.7802335651398216)

In [11]:
lightBGM = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LGBMClassifier(
        class_weight="balanced",
        num_leaves = 50,
        random_state=42
    ))
])
evaluate_model(lightBGM, X_train, y_train, X_test, y_test)

[LightGBM] [Info] Number of positive: 6981, number of negative: 33019
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.006566 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1550
[LightGBM] [Info] Number of data points in the train set: 40000, number of used features: 72
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000


C:\Users\vedik\AppData\Roaming\Python\Python311\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Train ROC-AUC: 0.8841
Test ROC-AUC : 0.7779

Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.76      0.83      8255
           1       0.37      0.64      0.47      1745

    accuracy                           0.74     10000
   macro avg       0.64      0.70      0.65     10000
weighted avg       0.81      0.74      0.77     10000


Confusion Matrix:
[[6313 1942]
 [ 626 1119]]


C:\Users\vedik\AppData\Roaming\Python\Python311\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


np.float64(0.77786243988622)

In [12]:
lightBGM = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LGBMClassifier(
        class_weight="balanced",
        num_leaves = 70,
        random_state=42
    ))
])
evaluate_model(lightBGM, X_train, y_train, X_test, y_test)

[LightGBM] [Info] Number of positive: 6981, number of negative: 33019
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.008859 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1550
[LightGBM] [Info] Number of data points in the train set: 40000, number of used features: 72
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000


C:\Users\vedik\AppData\Roaming\Python\Python311\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Train ROC-AUC: 0.9108
Test ROC-AUC : 0.7763

Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.78      0.84      8255
           1       0.37      0.62      0.47      1745

    accuracy                           0.75     10000
   macro avg       0.64      0.70      0.65     10000
weighted avg       0.81      0.75      0.77     10000


Confusion Matrix:
[[6422 1833]
 [ 657 1088]]


C:\Users\vedik\AppData\Roaming\Python\Python311\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


np.float64(0.7762636172572323)

| Parameter           | Train ROC-AUC        | Test ROC-AUC             | Confusion Matrix|
| ------------------- | ---------------- | ------------------- |------------------- |
| num_leaves = 15        | 0.8130 | 0.785956         | [[6083 2172] [ 547 1198]] |
| num_leaves = 31        | 0.8490    | 0.7802   | [[6209 2046] [ 600 1145]] |
| num_leaves = 50        | 0.8841    | 0.7779  | [[6313 1942] [ 626 1119]] |
| num_leaves = 70        | 0.9108    | 0.7763  | [[6422 1833] [ 657 1088]] |

- num_leaves = 15, random_state = 42 seems to be the best balance between train and test performance. Higher values show signs of overfitting.

### 2. Try max_depth = [-1, 5, 8, 12]
Now num_leaves = 15, random_state = 42 is fixed

In [13]:
lightBGM = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LGBMClassifier(
        class_weight="balanced",
        num_leaves = 70,
        random_state=42,
        max_depth = 5
    ))
])
evaluate_model(lightBGM, X_train, y_train, X_test, y_test)

[LightGBM] [Info] Number of positive: 6981, number of negative: 33019
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.015001 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1550
[LightGBM] [Info] Number of data points in the train set: 40000, number of used features: 72
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain

C:\Users\vedik\AppData\Roaming\Python\Python311\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Train ROC-AUC: 0.8282
Test ROC-AUC : 0.7827

Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.75      0.82      8255
           1       0.36      0.67      0.47      1745

    accuracy                           0.73     10000
   macro avg       0.64      0.71      0.64     10000
weighted avg       0.82      0.73      0.76     10000


Confusion Matrix:
[[6177 2078]
 [ 583 1162]]


C:\Users\vedik\AppData\Roaming\Python\Python311\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


np.float64(0.7826906329236948)

In [14]:
lightBGM = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LGBMClassifier(
        class_weight="balanced",
        num_leaves = 70,
        random_state=42,
        max_depth = -1
    ))
])
evaluate_model(lightBGM, X_train, y_train, X_test, y_test)

[LightGBM] [Info] Number of positive: 6981, number of negative: 33019
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.045546 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1550
[LightGBM] [Info] Number of data points in the train set: 40000, number of used features: 72
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000


C:\Users\vedik\AppData\Roaming\Python\Python311\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Train ROC-AUC: 0.9108
Test ROC-AUC : 0.7763

Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.78      0.84      8255
           1       0.37      0.62      0.47      1745

    accuracy                           0.75     10000
   macro avg       0.64      0.70      0.65     10000
weighted avg       0.81      0.75      0.77     10000


Confusion Matrix:
[[6422 1833]
 [ 657 1088]]


C:\Users\vedik\AppData\Roaming\Python\Python311\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


np.float64(0.7762636172572323)

In [16]:
lightBGM = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LGBMClassifier(
        class_weight="balanced",
        num_leaves = 15,
        random_state=42,
        max_depth = 8
    ))
])
evaluate_model(lightBGM, X_train, y_train, X_test, y_test)

[LightGBM] [Info] Number of positive: 6981, number of negative: 33019
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.010425 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1550
[LightGBM] [Info] Number of data points in the train set: 40000, number of used features: 72
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000


C:\Users\vedik\AppData\Roaming\Python\Python311\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\vedik\AppData\Roaming\Python\Python311\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Train ROC-AUC: 0.8126
Test ROC-AUC : 0.7853

Classification Report:
              precision    recall  f1-score   support

           0       0.92      0.73      0.81      8255
           1       0.35      0.69      0.46      1745

    accuracy                           0.72     10000
   macro avg       0.63      0.71      0.64     10000
weighted avg       0.82      0.72      0.75     10000


Confusion Matrix:
[[6042 2213]
 [ 547 1198]]


np.float64(0.7852967117263305)

In [17]:
lightBGM = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LGBMClassifier( max_depth = 12,
        class_weight="balanced",    
        num_leaves = 15,
        random_state=42
    ))
])
evaluate_model(lightBGM, X_train, y_train, X_test, y_test)

[LightGBM] [Info] Number of positive: 6981, number of negative: 33019
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.022171 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1550
[LightGBM] [Info] Number of data points in the train set: 40000, number of used features: 72
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000


C:\Users\vedik\AppData\Roaming\Python\Python311\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Train ROC-AUC: 0.8130
Test ROC-AUC : 0.7860

Classification Report:
              precision    recall  f1-score   support

           0       0.92      0.74      0.82      8255
           1       0.36      0.69      0.47      1745

    accuracy                           0.73     10000
   macro avg       0.64      0.71      0.64     10000
weighted avg       0.82      0.73      0.76     10000


Confusion Matrix:
[[6083 2172]
 [ 547 1198]]


C:\Users\vedik\AppData\Roaming\Python\Python311\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


np.float64(0.7859561713921752)

In [19]:
lightBGM = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LGBMClassifier( 
        class_weight="balanced",    
        num_leaves = 15,
        random_state=42,
        max_depth = 30
    ))
])
evaluate_model(lightBGM, X_train, y_train, X_test, y_test)

[LightGBM] [Info] Number of positive: 6981, number of negative: 33019
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.008488 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1550
[LightGBM] [Info] Number of data points in the train set: 40000, number of used features: 72
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000


C:\Users\vedik\AppData\Roaming\Python\Python311\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\vedik\AppData\Roaming\Python\Python311\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Train ROC-AUC: 0.8130
Test ROC-AUC : 0.7860

Classification Report:
              precision    recall  f1-score   support

           0       0.92      0.74      0.82      8255
           1       0.36      0.69      0.47      1745

    accuracy                           0.73     10000
   macro avg       0.64      0.71      0.64     10000
weighted avg       0.82      0.73      0.76     10000


Confusion Matrix:
[[6083 2172]
 [ 547 1198]]


np.float64(0.7859561713921752)

num_leaves = 15, random_state = 42 is fixed
| Parameter           | Train ROC-AUC        | Test ROC-AUC             | Confusion Matrix|
| ------------------- | ---------------- | ------------------- |------------------- |
| max_depth = 5      | 0.8282 | 0.7827        | [[6177 2078] [ 583 1162]] |
| max_depth = -1       | 0.9108   |  0.7763  | [[6422 1833] [ 657 1088]] |
| max_depth = 8        | 0.8126    | 0.7853 | [[6042 2213] [ 547 1198]] |
| max_depth = 12       | 0.8130    | 0.7860  | [[6083 2172] [ 547 1198]] |
| max_depth = 30       | 0.8130    | 0.7860  | [[6083 2172] [ 547 1198]] |

- max_depth = 12 seems to provide the best balance between train and test performance. Higher values show signs of overfitting, while -1 (no limit) leads to significant overfitting.
- num_leaves = 15, max_depth = 12, random_state = 42 is the best combination so far.

### 3. Tune Learning Rate + n_estimators Together


lightBGM = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LGBMClassifier( 
        class_weight="balanced",    
        num_leaves = 15,
        random_state=42,
        max_depth = 30
    ))
])
evaluate_model(lightBGM, X_train, y_train, X_test, y_test)

In [23]:
# 1. LR = 0.1, n_estimators = 200
lightBGM = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LGBMClassifier( 
        class_weight="balanced",    
        num_leaves = 15,
        random_state=42,
        max_depth = 30,
        learning_rate = 0.1,
        n_estimators = 200
    ))
])
evaluate_model(lightBGM, X_train, y_train, X_test, y_test)

[LightGBM] [Info] Number of positive: 6981, number of negative: 33019
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.017113 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1550
[LightGBM] [Info] Number of data points in the train set: 40000, number of used features: 72
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000


C:\Users\vedik\AppData\Roaming\Python\Python311\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Train ROC-AUC: 0.8398
Test ROC-AUC : 0.7834

Classification Report:
              precision    recall  f1-score   support

           0       0.92      0.75      0.82      8255
           1       0.36      0.67      0.47      1745

    accuracy                           0.73     10000
   macro avg       0.64      0.71      0.65     10000
weighted avg       0.82      0.73      0.76     10000


Confusion Matrix:
[[6167 2088]
 [ 570 1175]]


C:\Users\vedik\AppData\Roaming\Python\Python311\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


np.float64(0.7833652609601891)

In [24]:
# LR=0.05, n_estimator = 400
lightBGM = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LGBMClassifier( 
        class_weight="balanced",    
        num_leaves = 15,
        random_state=42,
        max_depth = 30,
        learning_rate = 0.05,
        n_estimators = 400
    ))
])
evaluate_model(lightBGM, X_train, y_train, X_test, y_test)

[LightGBM] [Info] Number of positive: 6981, number of negative: 33019
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.007065 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1550
[LightGBM] [Info] Number of data points in the train set: 40000, number of used features: 72
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000


C:\Users\vedik\AppData\Roaming\Python\Python311\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\vedik\AppData\Roaming\Python\Python311\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Train ROC-AUC: 0.8403
Test ROC-AUC : 0.7836

Classification Report:
              precision    recall  f1-score   support

           0       0.92      0.75      0.82      8255
           1       0.36      0.68      0.47      1745

    accuracy                           0.74     10000
   macro avg       0.64      0.71      0.65     10000
weighted avg       0.82      0.74      0.76     10000


Confusion Matrix:
[[6178 2077]
 [ 562 1183]]


np.float64(0.7835905303549642)

In [25]:
# LR = 0.03, n_estimators = 600
lightBGM = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LGBMClassifier( 
        class_weight="balanced",    
        num_leaves = 15,
        random_state=42,
        max_depth = 30,
        learning_rate = 0.03,
        n_estimators = 600
    ))
])
evaluate_model(lightBGM, X_train, y_train, X_test, y_test)

[LightGBM] [Info] Number of positive: 6981, number of negative: 33019
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.007215 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1550
[LightGBM] [Info] Number of data points in the train set: 40000, number of used features: 72
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000


C:\Users\vedik\AppData\Roaming\Python\Python311\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\vedik\AppData\Roaming\Python\Python311\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Train ROC-AUC: 0.8353
Test ROC-AUC : 0.7839

Classification Report:
              precision    recall  f1-score   support

           0       0.92      0.75      0.82      8255
           1       0.36      0.67      0.47      1745

    accuracy                           0.74     10000
   macro avg       0.64      0.71      0.65     10000
weighted avg       0.82      0.74      0.76     10000


Confusion Matrix:
[[6182 2073]
 [ 573 1172]]


np.float64(0.7838558553555282)

num_leaves = 15, random_state = 42, max_depth = 12 is fixed
| Parameter           | Train ROC-AUC        | Test ROC-AUC             | Confusion Matrix|
| ------------------- | ---------------- | ------------------- |------------------- |
| LR=0.1, n_estimator = 200   | 0.8398 | 0.7834       | [[6167 2088] [ 570 1175]]|
| LR=0.05, n_estimator = 400      |   0.8353  |  0.7839 | [[6178 2077] [ 562 1183]] |
| LR = 0.03, n_estimators = 600      | 0.8126    | 0.7853 | [[6182 2073] [ 573 1172]] |

- LR=0.03, n_estimators = 600 seems to provide the best balance between train and test performance. Higher learning rates show signs of underfitting, while too low learning rates with high n_estimators can lead to overfitting.

- num_leaves = 15, random_state = 42, max_depth = 12 is fixed no need of LR and n_estimators tuning as it is not improving the performance. We will stick to num_leaves = 15, max_depth = 12, random_state = 42 as our final model. 

In [33]:

lightBGM = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LGBMClassifier( 
        class_weight="balanced",    
        num_leaves = 15,
        random_state=42,
        max_depth = 12
    ))
])
evaluate_model(lightBGM, X_train, y_train, X_test, y_test)

[LightGBM] [Info] Number of positive: 6981, number of negative: 33019
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.008732 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1550
[LightGBM] [Info] Number of data points in the train set: 40000, number of used features: 72
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000


C:\Users\vedik\AppData\Roaming\Python\Python311\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Train ROC-AUC: 0.8130
Test ROC-AUC : 0.7860

Classification Report:
              precision    recall  f1-score   support

           0       0.92      0.74      0.82      8255
           1       0.36      0.69      0.47      1745

    accuracy                           0.73     10000
   macro avg       0.64      0.71      0.64     10000
weighted avg       0.82      0.73      0.76     10000


Confusion Matrix:
[[6083 2172]
 [ 547 1198]]


C:\Users\vedik\AppData\Roaming\Python\Python311\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


np.float64(0.7859561713921752)

### 3. Try min_child_samples = [20, 40, 60, 100]

In [32]:

lightBGM = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LGBMClassifier( 
        class_weight="balanced",    
        num_leaves = 15,
        random_state=42,
        max_depth = 12,
        min_child_samples=20
    ))
])
evaluate_model(lightBGM, X_train, y_train, X_test, y_test)

[LightGBM] [Info] Number of positive: 6981, number of negative: 33019
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.034864 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1550
[LightGBM] [Info] Number of data points in the train set: 40000, number of used features: 72
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000


C:\Users\vedik\AppData\Roaming\Python\Python311\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Train ROC-AUC: 0.8130
Test ROC-AUC : 0.7860

Classification Report:
              precision    recall  f1-score   support

           0       0.92      0.74      0.82      8255
           1       0.36      0.69      0.47      1745

    accuracy                           0.73     10000
   macro avg       0.64      0.71      0.64     10000
weighted avg       0.82      0.73      0.76     10000


Confusion Matrix:
[[6083 2172]
 [ 547 1198]]


C:\Users\vedik\AppData\Roaming\Python\Python311\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


np.float64(0.7859561713921752)

In [34]:

lightBGM = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LGBMClassifier( 
        class_weight="balanced",    
        num_leaves = 15,
        random_state=42,
        max_depth = 12,
        min_child_samples=40
    ))
])
evaluate_model(lightBGM, X_train, y_train, X_test, y_test)

[LightGBM] [Info] Number of positive: 6981, number of negative: 33019
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.055645 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1550
[LightGBM] [Info] Number of data points in the train set: 40000, number of used features: 72
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000


C:\Users\vedik\AppData\Roaming\Python\Python311\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Train ROC-AUC: 0.8127
Test ROC-AUC : 0.7855

Classification Report:
              precision    recall  f1-score   support

           0       0.92      0.73      0.82      8255
           1       0.35      0.69      0.47      1745

    accuracy                           0.73     10000
   macro avg       0.64      0.71      0.64     10000
weighted avg       0.82      0.73      0.75     10000


Confusion Matrix:
[[6065 2190]
 [ 548 1197]]


C:\Users\vedik\AppData\Roaming\Python\Python311\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


np.float64(0.7854844940723604)

In [ ]:

lightBGM = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LGBMClassifier( 
        class_weight="balanced",    
        num_leaves = 15,
        random_state=42,
        max_depth = 12,
        min_child_samples=100
    ))
])
evaluate_model(lightBGM, X_train, y_train, X_test, y_test)

[LightGBM] [Info] Number of positive: 6981, number of negative: 33019
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.009621 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1550
[LightGBM] [Info] Number of data points in the train set: 40000, number of used features: 72
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000


C:\Users\vedik\AppData\Roaming\Python\Python311\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Train ROC-AUC: 0.8109
Test ROC-AUC : 0.7866

Classification Report:
              precision    recall  f1-score   support

           0       0.92      0.73      0.82      8255
           1       0.35      0.69      0.47      1745

    accuracy                           0.73     10000
   macro avg       0.64      0.71      0.64     10000
weighted avg       0.82      0.73      0.75     10000


Confusion Matrix:
[[6062 2193]
 [ 545 1200]]


C:\Users\vedik\AppData\Roaming\Python\Python311\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


np.float64(0.7865557559107184)

min_child_sample = 100 gives :- Train ROC-AUC: 0.8109, 
Test ROC-AUC : 0.7866

- So final struct:  class_weight="balanced",    
        num_leaves = 15,
        random_state=42,
        max_depth = 12,
        min_child_samples=100